# Data Loading and QC

This notebook uses the shared `rarecell` modules to load a CITE-seq dataset, validate modalities, and write QC summaries for the benchmark workflow.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from rarecell.config import FIGURES_DIR, TABLES_DIR
from rarecell.io import (
    get_cell_labels,
    get_protein_matrix,
    get_rna_adata,
    load_citeseq,
    validate_citeseq_object,
)
from rarecell.plotting import plot_bar_counts, plot_histogram, plot_qc_summary
from rarecell.qc import (
    make_candidate_target_population_table,
    make_dataset_summary,
    make_protein_feature_table,
    make_protein_qc_cell_table,
    make_rna_qc_cell_table,
    save_summary_tables,
    summarize_cell_labels,
)
from rarecell.utils import write_json

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

Set `input_spec` to one supported input. The shared `load_citeseq` helper handles local files or scvi 10x specs.

In [2]:
input_spec = "scvi:5k_pbmc_protein_v3_nextgem"
label_key = None

In [3]:
obj = load_citeseq(input_spec)
validation = validate_citeseq_object(obj)
write_json(validation, TABLES_DIR / "dataset_validation.json")
pd.DataFrame([validation]).to_csv(TABLES_DIR / "dataset_validation.csv", index=False)
validation

{'object_type': 'AnnData',
 'modalities': ['rna', 'protein'],
 'rna_key': 'X',
 'protein_key': 'feature_types',
 'protein_location': 'var',
 'n_cells': 5527,
 'n_genes': 33538,
 'protein_modality_exists': True,
 'n_protein_features': 32,
 'protein_error': None,
 'cell_type_labels_exist': False,
 'cell_type_label_key': None,
 'candidate_label_columns': [],
 'barcodes_align': True}

In [4]:
if not validation["protein_modality_exists"]:
    raise ValueError(validation.get("protein_error") or "No protein/ADT modality was found.")

rna = get_rna_adata(obj)
protein = get_protein_matrix(obj)
labels = get_cell_labels(obj, preferred_keys=[label_key] if label_key else None)

summary = make_dataset_summary(obj)
summary["cell_type_counts"] = summarize_cell_labels(labels)
save_summary_tables(summary, TABLES_DIR)

rna_qc = make_rna_qc_cell_table(rna)
protein_qc = make_protein_qc_cell_table(protein)
protein_features = make_protein_feature_table(protein)
rna_qc.to_csv(TABLES_DIR / "rna_qc_cells.csv", index=False)
protein_qc.to_csv(TABLES_DIR / "protein_qc_cells.csv", index=False)
protein_features.to_csv(TABLES_DIR / "protein_features.csv", index=False)

if labels is not None:
    candidates = make_candidate_target_population_table(labels)
    candidates.to_csv(TABLES_DIR / "candidate_target_populations.csv", index=False)
else:
    candidates = pd.DataFrame()

plot_qc_summary(summary, FIGURES_DIR / "qc_summary.png")
plot_histogram(rna_qc["total_counts"], FIGURES_DIR / "rna_total_counts_hist.png",
               "Distribution of RNA UMI counts per cell", "RNA UMI counts per cell")
plot_histogram(rna_qc["n_genes_by_counts"], FIGURES_DIR / "rna_genes_per_cell_hist.png",
               "Distribution of detected genes per cell", "Detected genes per cell")
plot_histogram(protein_qc["total_protein_counts"], FIGURES_DIR / "protein_total_counts_hist.png",
               "Distribution of antibody-derived tag counts per cell", "ADT counts per cell")
if labels is not None:
    plot_bar_counts(labels, FIGURES_DIR / "cell_label_counts.png",
                    "Cell counts by annotated cell type", "Cell type", "Number of cells")

In [5]:
display(summary["dataset_summary"])
display(summary["protein_summary"])
display(protein_features.head())
display(rna_qc.head())
display(protein_qc.head())
if labels is not None:
    display(summary["cell_type_counts"].head())
    display(candidates.head())
else:
    print("No cell labels found. The next step can fall back to Leiden clusters as candidate populations.\n")


,n_cells,n_genes,total_counts_mean,total_counts_median,genes_per_cell_mean,genes_per_cell_median,detected_cells_per_gene_mean
0,5527,33538,5990.152832,5676.0,1795.484892,1836.0,295.89257


,n_cells,n_proteins,total_counts_mean,total_counts_median,zero_fraction
0,5527,32,2936.567139,2513.0,0.081599


,protein_name,total_counts,mean_counts,detected_cells,detected_fraction
0,CD3_TotalSeqB,1736565.0,314.196686,5517,0.998191
1,CD4_TotalSeqB,2957270.0,535.058777,5510,0.996924
2,CD8a_TotalSeqB,243254.0,44.011940,5415,0.979736
3,CD11b_TotalSeqB,3384634.0,612.381775,5524,0.999457
4,CD14_TotalSeqB,1191379.0,215.556183,5525,0.999638


,cell_id,total_counts,n_genes_by_counts,pct_counts_mt
0,AAACCCACAGGCTTGC-1,4428.0,1685,NaN
1,AAACCCAGTAGTTAGA-1,7107.0,2291,NaN
2,AAACGAAGTAACGATA-1,11060.0,3186,NaN
3,AAACGAAGTGGATCAG-1,4838.0,1850,NaN
4,AAACGAATCATGAGAA-1,4244.0,1410,NaN


,cell_id,total_protein_counts,detected_proteins,zero_fraction_per_cell
0,AAACCCACAGGCTTGC-1,4761.0,31,0.03125
1,AAACCCAGTAGTTAGA-1,4010.0,30,0.06250
2,AAACGAAGTAACGATA-1,1824.0,29,0.09375
3,AAACGAAGTGGATCAG-1,3144.0,30,0.06250
4,AAACGAATCATGAGAA-1,3118.0,31,0.03125


No cell labels found. The next step can fall back to Leiden clusters as candidate populations.



## Summary

In [6]:
# Summary of generated outputs and deviations from make_qc_summary.py
generated = [
    (TABLES_DIR / "dataset_validation.json", "notebook-only"),
    (TABLES_DIR / "dataset_validation.csv", "notebook-only"),
    (TABLES_DIR / "dataset_summary.csv", ""),
    (TABLES_DIR / "rna_qc_cells.csv", ""),
    (TABLES_DIR / "protein_qc_cells.csv", ""),
    (TABLES_DIR / "protein_features.csv", ""),
    (TABLES_DIR / "candidate_target_populations.csv", "if labels present"),
    (FIGURES_DIR / "qc_summary.png", ""),
    (FIGURES_DIR / "rna_total_counts_hist.png", ""),
    (FIGURES_DIR / "rna_genes_per_cell_hist.png", ""),
    (FIGURES_DIR / "protein_total_counts_hist.png", ""),
    (FIGURES_DIR / "cell_label_counts.png", "if labels present"),
]
print("Generated outputs:")
for p, note in generated:
    status = "OK" if p.exists() else "MISSING"
    suffix = f"  # {note}" if note else ""
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = p
    print(f"  [{status}] {rel}{suffix}")
print()
print("Deviations from make_qc_summary.py:")
print("  + Writes dataset_validation.json/csv (notebook-only interactive validation).")
print("  - No file-level logging (script uses setup_file_logger).")


Generated outputs:
  [OK] results/tables/dataset_validation.json  # notebook-only
  [OK] results/tables/dataset_validation.csv  # notebook-only
  [OK] results/tables/dataset_summary.csv
  [OK] results/tables/rna_qc_cells.csv
  [OK] results/tables/protein_qc_cells.csv
  [OK] results/tables/protein_features.csv
  [OK] results/tables/candidate_target_populations.csv  # if labels present
  [OK] results/figures/qc_summary.png
  [OK] results/figures/rna_total_counts_hist.png
  [OK] results/figures/rna_genes_per_cell_hist.png
  [OK] results/figures/protein_total_counts_hist.png
  [MISSING] results/figures/cell_label_counts.png  # if labels present

Deviations from make_qc_summary.py:
  + Writes dataset_validation.json/csv (notebook-only interactive validation).
  - No file-level logging (script uses setup_file_logger).
